# Machine Learning for Environmental Time-Series Forecasting
### 1-Day-Ahead Reference Evapotranspiration Forecasting with XGBoost and MVMD

**A 90-minute guided lab · Davis CIMIS station · Run one cell at a time**

Reference evapotranspiration (**ETo**) describes the water that would leave a reference surface through evaporation and plant transpiration. A forecast can help anticipate irrigation demand. Here, ETo is measured in **mm/day**.

A **time series** is a set of observations recorded in time order. Weather today and on previous days can provide clues about tomorrow. We will build two models that predict tomorrow's ETo from meteorological observations available through today.

> **Our boundary:** ETo is the target only. Past ETo never enters either model as a predictor, and ETo is never decomposed.

```text
Raw Environmental Data
          ↓
Check Data Quality → Fill Missing Values
          ↓
Identify Useful Lags (XCF / PXCF)
          ↓
Baseline XGBoost
          ↓
MVMD Decomposition → MVMD + XGBoost
          ↓
Compare Forecasts on the Same Dates
```



## 0. Get ready

We will import familiar Python tools and install XGBoost or statsmodels **only if they are missing**. Expect a short version table; it helps explain small differences between computers. Next, we upload the workshop CSV.

In Google Colab, the usual NumPy, pandas, matplotlib, and scikit-learn tools are already available. A CPU runtime is enough; no MATLAB or GPU is required.

In [ ]:
#### pip install
import importlib.util
import subprocess
import sys
for package in ["xgboost", "statsmodels"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import ccf
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from IPython.display import display
from importlib.metadata import version

plt.rcParams.update({
    "figure.figsize": (12, 4.5), "font.size": 12,
    "axes.titlesize": 15, "axes.labelsize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.18,
    "figure.dpi": 110, "savefig.dpi": 150,
})
colors = ["#0072B2", "#D55E00", "#009E73", "#CC79A7"]
display(pd.DataFrame({"Package": ["numpy", "pandas", "scikit-learn", "statsmodels", "xgboost"],
                      "Version": [version(p) for p in ["numpy", "pandas", "scikit-learn", "statsmodels", "xgboost"]]}))

## 1. Load and explore the data

“We first need to understand what is inside our dataset before building a machine-learning model.”

Upload **Davis_ETo_Workshop.csv** below. We will check its columns, read dates, and put the rows in time order. Expect the first five rows and a brief dataset summary. These checks prepare us to look for missing observations.

**Local use:** place the CSV beside this notebook and run it there. Alternatively, set `LOCAL_CSV` to your own file path. Colab users should leave it as `None`; no instructor directory is needed.

In [ ]:
# Load Davis Station Dataset from a publicly shared CSV (Google Colab)
# No Google Drive mount or Google account authorization required.
from io import BytesIO
from pathlib import Path
import requests

# Paste the SHARE LINK of your CSV file here (not the notebook link).
# Set CSV sharing to: General access > Anyone with the link > Viewer.
CSV_SHARE_LINK = "https://drive.google.com/file/d/1nf9v0U_Cpa90-Ppb9WRsnphlZZYBSerG/view?usp=sharing"

if CSV_SHARE_LINK == "PASTE_PUBLIC_CSV_GOOGLE_DRIVE_LINK_HERE":
    raise ValueError(
        "Paste the public sharing link for your CSV into CSV_SHARE_LINK. "
        "This must be the CSV link, not the notebook link."
    )

# Extract the file ID from standard Google Drive sharing links.
import re
match = re.search(r"/file/d/([A-Za-z0-9_-]+)", CSV_SHARE_LINK)
if not match:
    match = re.search(r"[?&]id=([A-Za-z0-9_-]+)", CSV_SHARE_LINK)
if not match:
    raise ValueError("Could not find a Google Drive file ID in CSV_SHARE_LINK.")

file_id = match.group(1)
csv_url = f"https://drive.google.com/uc?export=download&id={file_id}"
response = requests.get(csv_url, timeout=60)
response.raise_for_status()
if b"<html" in response.content[:1000].lower() or b"<!doctype html" in response.content[:1000].lower():
    raise ValueError(
        "Google Drive returned a webpage instead of CSV data. "
        "Make sure the CSV is shared as 'Anyone with the link > Viewer'. "
        "Some large files may require a different direct-download method."
    )
file_path = Path("Davis_ETo_Workshop.csv")
csv_data = BytesIO(response.content)

target = "ETo_mm"
predictors = ["Solar_Radiation_Wm2", "Air_Temp_C", "Relative_Humidity_pct", "Wind_Speed_ms"]
variable_names = dict(zip(predictors, ["Solar radiation", "Air temperature", "Relative humidity", "Wind speed"]))
required = ["Date", target] + predictors

raw_df = pd.read_csv(csv_data)
raw_df = raw_df[required].copy()
raw_df["Date"] = pd.to_datetime(raw_df["Date"], errors="coerce")
raw_df = raw_df.sort_values("Date").reset_index(drop=True)
for col in [target] + predictors:
    raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

print(f"Loaded {len(raw_df):,} daily records from: {file_path.name}")
print(f"Target (Output): {target} | Predictors (Inputs): {predictors}")
display(raw_df.head())

### What each column means

| Column | Meaning | Role |
|:--|:--|:--|
| Date | Day of observation | Time index |
| ETo_mm | Reference evapotranspiration, mm/day | **Target only** |
| Solar_Radiation_Wm2 | Solar radiation, W/m² | Predictor |
| Air_Temp_C | Air temperature, °C | Predictor |
| Relative_Humidity_pct | Relative humidity, % | Predictor |
| Wind_Speed_ms | Wind speed, m/s | Predictor |

We will summarize the size, coverage, data types, and typical values. Expect numerical ranges in different units. The next step checks which values are absent; the `count` column below may already hint at gaps.


### Explore the Dataset

Before preparing our data for machine learning, let's examine its basic characteristics.

In this step, we will:

1. **Check the dataset size:** Identify the total number of observations and columns.
2. **Check the time period:** Find the first and last dates in our dataset.
3. **Identify data types:** Check whether each variable contains dates, integers, or decimal numbers.
4. **Explore descriptive statistics:** Examine the count, mean, standard deviation, minimum, maximum, and quartiles of each variable.
5. **Look for possible missing values:** Compare the number of available observations across variables.

**Why is this important?**

Understanding the dataset helps us identify potential data-quality issues before preprocessing and training our forecasting models.


In [ ]:
print(f"Dataset shape: {raw_df.shape[0]:,} rows × {raw_df.shape[1]} columns")
print(f"Start: {raw_df.Date.min():%Y-%m-%d} | End: {raw_df.Date.max():%Y-%m-%d}")
display(raw_df.dtypes.astype(str).rename("Data type").to_frame())
display(raw_df[[target] + predictors].describe().T.round(2))

## 2. Check for missing data

Environmental datasets often contain missing observations because of sensor problems, communication failures, maintenance, or quality-control filtering. Many analysis methods require complete numbers. XGBoost can handle some missing inputs, but our correlation calculations and MVMD need a consistent, complete signal.

We will count missing entries and their percentage in each column. Expect a compact table, including zeros where data are complete. This tells us whether gap filling is needed next.

In [ ]:
missing_summary = pd.DataFrame({
    "Missing count": raw_df.isna().sum(),
    "Missing (%)": raw_df.isna().mean().mul(100).round(2),
})
display(missing_summary)
print(f"Total missing measurements: {raw_df[[target] + predictors].isna().sum().sum()}")

## 3. Fill missing values using KNN Imputation


1. **Create a copy of the dataset:** Keep the original data unchanged.
2. **Check missing values:** Count missing observations in each variable.
3. **Apply KNN imputation:** Use the K-Nearest Neighbors method to estimate missing values based on similar observations.
4. **Use two nearest neighbors:** Estimate each missing value using the average of its two closest observations.
5. **Verify the results:** Check that no missing values remain after imputation.








In [ ]:

from sklearn.impute import KNNImputer

# Create a copy of the original dataset
df = raw_df.copy()

# Select the measurement columns
measurement_columns = [target] + predictors

# Check missing values before imputation
print("Missing values before KNN:")
print(df[measurement_columns].isna().sum())

# Apply KNN imputation
imputer = KNNImputer(n_neighbors=2, weights="uniform")

df[measurement_columns] = imputer.fit_transform(
    df[measurement_columns]
)

# Check missing values after imputation
print("\nMissing values after KNN:")
print(df[measurement_columns].isna().sum())

# Use the imputed data for the workshop
forecast_df = df.copy()

# Verify results
assert df[measurement_columns].notna().all().all()
assert df.Date.equals(raw_df.Date)

print("\nKNN imputation completed successfully.")


## 4. See ETo through time

Each point represents a daily observation. Environmental time series can contain rapid changes and seasonal patterns. We will draw one ETo time-series graph using the exploration copy. Expect time along the horizontal axis and ETo along the vertical axis.

Look for repeated rises and falls. **Time order matters**: that is why the next steps examine earlier weather and preserve chronological order.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.Date, df[target], color=colors[0], lw=1)
ax.set(title="Daily reference evapotranspiration — Davis CIMIS", xlabel="Date", ylabel="ETo (mm/day)")
fig.tight_layout()
plt.show()

## 5. Lag identification — Cross-Correlation (XCF)

Weather conditions from previous days may influence ETo forecasting. Before building our models, we need to identify which historical observations contain useful information.

**What is XCF?**

Cross-correlation (XCF) measures the relationship between ETo and each meteorological variable at different time steps.

In this step, we will:

1. **Examine historical observations:** Analyze time steps from **t, t−1, t−2, ..., t−12**, where t represents the current day.
2. **Calculate cross-correlations:** Measure the relationship between ETo and solar radiation, air temperature, relative humidity, and wind speed.
3. **Create bar charts:** Visualize the cross-correlation values for each meteorological variable.


**How to interpret the graphs**

- **Positive bars:** Indicate a positive relationship with ETo.
- **Negative bars:** Indicate a negative relationship with ETo.
- **Bars beyond ±0.10:** Indicate relationships that exceed our selected exploratory threshold.
- **Bars close to zero:** Indicate weak linear relationships.

**Why is this important?**

XCF helps us understand how weather conditions from previous days relate to ETo and supports the selection of historical inputs for our forecasting models.

**Next:** We will use Partial Cross-Correlation (PXCF) to examine these relationships more carefully.


In [ ]:
max_lag = 12
threshold = 0.10
# XCF analysis using KNN-imputed data

lags = np.arange(max_lag + 1)
xcf_results = {}

# Calculate XCF for each meteorological variable
for variable in predictors:
    xcf_results[variable] = [
        forecast_df[target].corr(forecast_df[variable].shift(lag))
        for lag in lags
    ]

# Plot XCF bar chart
bar_width = 0.18
offsets = np.linspace(-0.27, 0.27, len(predictors))

fig, ax = plt.subplots(figsize=(12, 5))

for i, variable in enumerate(predictors):
    ax.bar(
        lags + offsets[i],
        xcf_results[variable],
        width=bar_width,
        label=variable_names[variable],
        color=colors[i]
    )

# Correlation thresholds
ax.axhline(threshold, linestyle="--", color="gray",
           label="Threshold ±0.10")
ax.axhline(-threshold, linestyle="--", color="gray")
ax.axhline(0, color="black", linewidth=0.8)

# Figure settings
ax.set_xticks(lags)
ax.set_xticklabels(["t"] + [f"t−{i}" for i in lags[1:]])
ax.set_xlabel("Weather Time Lag (days)")
ax.set_ylabel("Cross-Correlation Coefficient")
ax.set_ylim(-1, 1)
ax.set_title("XCF — Meteorological Variables and ETo (KNN-Imputed Data)")

ax.legend(ncol=3, fontsize=10)
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

# Display correlation values
xcf_table = pd.DataFrame(xcf_results, index=lags)
display(xcf_table.round(3))



### Selecting the Time-Step Range for XCF and PXCF

The range of time steps we examine is a research decision based on our experience, scientific understanding, and the characteristics of the data.

In this workshop, we decided to examine **t through t−12**, meaning the current day and the previous 12 days.

This range allows us to explore how weather observations from different days are related to ETo.

Different researchers may choose different ranges depending on their data and research objectives.



## 6. Lag identification — Partial Cross-Correlation Analysis (PXCF)

**Objective:** Identify which past weather observations provide additional information for ETo forecasting.

### What is the difference between XCF and PXCF?

- **XCF (Cross-Correlation):** Measures the overall relationship between ETo and weather observations from different days.

- **PXCF (Partial Cross-Correlation):** Checks whether older weather observations still have a relationship with ETo after considering more recent observations of the same weather variable.

**For example:**

Suppose solar radiation at **t−3** shows a strong relationship with ETo.

- **XCF:** Shows that solar radiation at t−3 is related to ETo.
- **PXCF:** Checks whether solar radiation at t−3 provides additional information after considering solar radiation at t, t−1, and t−2.



### Why is this important?

PXCF helps us understand whether older observations provide additional information beyond more recent observations.

Together, XCF and PXCF help us select useful historical weather data for forecasting.



In [ ]:

# PXCF analysis using KNN-imputed data

from sklearn.linear_model import LinearRegression

pxcf_results = {}

# Calculate partial correlations
for variable in predictors:
    pxcf_results[variable] = []

    for lag in lags:

        # Create ETo and weather lag columns
        data = pd.DataFrame({
            "ETo": forecast_df[target],
            "Weather": forecast_df[variable].shift(lag)
        })

        if lag == 0:
            pxcf = data["ETo"].corr(data["Weather"])

        else:
            # Add shorter weather lags as control variables
            for k in range(lag):
                data[f"Lag_{k}"] = forecast_df[variable].shift(k)

            data = data.dropna()

            controls = data[[f"Lag_{k}" for k in range(lag)]]

            # Remove the effects of shorter lags
            model = LinearRegression()

            eto_residual = data["ETo"] - model.fit(
                controls, data["ETo"]
            ).predict(controls)

            weather_residual = data["Weather"] - model.fit(
                controls, data["Weather"]
            ).predict(controls)

            pxcf = eto_residual.corr(weather_residual)

        pxcf_results[variable].append(pxcf)

# Plot PXCF bar chart
fig, ax = plt.subplots(figsize=(12, 5))

for i, variable in enumerate(predictors):
    ax.bar(
        lags + offsets[i],
        pxcf_results[variable],
        width=bar_width,
        label=variable_names[variable],
        color=colors[i]
    )

# Correlation thresholds
ax.axhline(threshold, linestyle="--", color="gray",
           label="Threshold ±0.10")
ax.axhline(-threshold, linestyle="--", color="gray")
ax.axhline(0, color="black", linewidth=0.8)

# Figure settings
ax.set_xticks(lags)
ax.set_xticklabels(["t"] + [f"t−{i}" for i in lags[1:]])
ax.set_xlabel("Weather Time Lag (days)")
ax.set_ylabel("Partial Cross-Correlation Coefficient")
ax.set_ylim(-1, 1)
ax.set_title("PXCF — Meteorological Variables and ETo (KNN-Imputed Data)")

ax.legend(ncol=3, fontsize=10)
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

# Display partial correlation values
pxcf_table = pd.DataFrame(pxcf_results, index=lags)
display(pxcf_table.round(3))


## 7. Create model inputs

A lagged input is simply an earlier measurement: **Solar_Radiation(t−1)** means “solar radiation observed one day earlier.” We will make one column for every selected variable–lag pair and place **tomorrow's ETo** beside them as the answer to learn.

In [ ]:

# Prepare lagged meteorological data for XGBoost

max_lag = 5

# Tomorrow's ETo is the prediction target
forecast_target = raw_df[target].shift(-1)
forecast_dates = raw_df["Date"].shift(-1)

# Create modeling dataset
model_df = pd.DataFrame({
    "Origin_Date": raw_df["Date"],
    "Date": forecast_dates,
    target: forecast_target
})

# Create weather features from t to t-5
feature_columns = []

for variable in predictors:
    for lag in range(max_lag + 1):

        column_name = f"{variable}_t-{lag}"

        model_df[column_name] = forecast_df[variable].shift(lag)
        feature_columns.append(column_name)

# Remove rows without complete inputs or target
model_df = model_df.dropna(subset=[target] + feature_columns)

# Display results
print(f"Modeling observations: {len(model_df)}")
print(f"Number of input features: {len(feature_columns)}")
print("Input lags: t, t-1, t-2, t-3, t-4, t-5")
print("Prediction target: ETo at t+1")

display(model_df.head())


## 8. Chronological train/test split

Randomly shuffling days would mix future and past. Instead, we follow the source notebook: **first 75% for training, final 25% for testing**, with no shuffle. We train once, then forecast successive test days using weather observed through each origin.

```text
PAST ------------------------------------------------> FUTURE
|                TRAIN (75%)                | TEST (25%) |
```


In [ ]:

# Split data: 75% training and 25% testing

import matplotlib.pyplot as plt

# Ensure data are in chronological order
model_df = model_df.sort_values("Date").copy()

# Define split point
split_index = int(len(model_df) * 0.75)

# Split the dataset
train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()

# Separate inputs (X) and target (y)
X_train = train_df[feature_columns]
y_train = train_df[target]

X_test = test_df[feature_columns]
y_test = test_df[target]

# Display results
print(f"Training: {len(train_df)} observations (75%)")
print(f"Testing:  {len(test_df)} observations (25%)")

print("\nTraining period:",
      train_df["Date"].min().date(), "to",
      train_df["Date"].max().date())

print("Testing period:",
      test_df["Date"].min().date(), "to",
      test_df["Date"].max().date())

# Verify chronological split
assert train_df["Date"].max() < test_df["Date"].min()

# --------------------------------------------------
# Visualize the chronological training/testing split
# --------------------------------------------------

train_pct = len(train_df) / len(model_df) * 100
test_pct = len(test_df) / len(model_df) * 100

fig, ax = plt.subplots(figsize=(11, 2.5))

# Training section
ax.barh(
    0, train_pct, left=0, height=0.45,
    color="#237B91"
)

# Testing section
ax.barh(
    0, test_pct, left=train_pct, height=0.45,
    color="#D48A36"
)

# Labels inside the bars
ax.text(
    train_pct / 2, 0,
    f"Training ({train_pct:.1f}%)",
    ha="center", va="center",
    color="white", fontsize=13, fontweight="bold"
)

ax.text(
    train_pct + test_pct / 2, 0,
    f"Testing ({test_pct:.1f}%)",
    ha="center", va="center",
    color="white", fontsize=12, fontweight="bold"
)

# Mark split point
ax.axvline(train_pct, color="white", linewidth=3)

# Figure formatting
ax.set_xlim(0, 100)
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_xticks([0, 25, 50, 75, 100])
ax.set_xticklabels(["0%", "25%", "50%", "75%", "100%"])
ax.set_xlabel("Percentage of Observations", fontsize=11)
ax.set_title(
    "Chronological Train–Test Split",
    fontsize=14, fontweight="bold", pad=15
)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(axis="x", length=0)
ax.grid(False)

plt.tight_layout()
plt.show()


## 9. Standalone model — XGBoost

XGBoost combines many small decision trees. Each new tree focuses on correcting errors made by the previous trees. Here it learns a connection between our selected weather inputs and tomorrow's ETo.


In [ ]:

from xgboost import XGBRegressor

# Define the XGBoost model
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

# Train the model using 75% of the data
xgb_model.fit(X_train, y_train)

# Predict ETo using the remaining 25%
y_pred = xgb_model.predict(X_test)



### Measure the errors

We will compare predictions with measured test ETo. Expect one small table:

| Metric | Plain-language meaning | Preferred direction |
|:--|:--|:--|
| RMSE (mm/day) | Typical error with more weight on large misses | Lower |
| MAE (mm/day) | Average absolute miss | Lower |
| R² | Fit relative to the variation around the test mean | Higher; 1 is perfect |


In [ ]:

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

# Calculate evaluation metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Display results
results = pd.DataFrame({
    "Metric": ["RMSE (mm/day)", "MAE (mm/day)", "R²"],
    "Value": [rmse, mae, r2]
})

display(results.round(4))


### How close are predictions to observations?

We will draw a scatter plot with measured ETo on the horizontal axis and predicted ETo on the vertical axis. Each dot is one held-out day. Dots on the dashed **1:1 line** are exact predictions; dots above it are overpredictions and dots below it are underpredictions.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Predict ETo for training and testing datasets
y_train_pred = xgb_model.predict(X_train)
y_test_pred = xgb_model.predict(X_test)

# Convert observations to arrays
y_train_true = np.asarray(y_train).flatten()
y_test_true = np.asarray(y_test).flatten()

# Calculate Pearson correlation coefficients
R_train = np.corrcoef(y_train_true, y_train_pred)[0, 1]
R_test = np.corrcoef(y_test_true, y_test_pred)[0, 1]

# Create scatter plot
plt.figure(figsize=(8, 7))

plt.scatter(y_train_true, y_train_pred,
            color="blue", alpha=0.5, s=30,
            label="Training")

plt.scatter(y_test_true, y_test_pred,
            color="red", alpha=0.6, s=30,
            marker="x", label="Testing")

# Plot 1:1 reference line
min_val = min(y_train_true.min(), y_test_true.min(),
              y_train_pred.min(), y_test_pred.min())

max_val = max(y_train_true.max(), y_test_true.max(),
              y_train_pred.max(), y_test_pred.max())

plt.plot([min_val, max_val], [min_val, max_val],
         "k--", label="1:1 Line")

# Display correlation coefficients
plt.text(0.05, 0.92, f"R (Train) = {R_train:.3f}",
         transform=plt.gca().transAxes,
         fontsize=11, color="blue")

plt.text(0.05, 0.86, f"R (Test) = {R_test:.3f}",
         transform=plt.gca().transAxes,
         fontsize=11, color="red")

# Figure settings
plt.xlabel("Observed ETo (mm/day)")
plt.ylabel("Predicted ETo (mm/day)")
plt.title("Observed vs. Predicted ETo — XGBoost (1-Day Ahead)")

plt.xlim(min_val, max_val)
plt.ylim(min_val, max_val)
plt.gca().set_aspect("equal", adjustable="box")

plt.legend(loc="lower right")
plt.grid(alpha=0.2)
plt.tight_layout()

# Save figure
plt.savefig("XGBoost_Observed_vs_Predicted.png",
            dpi=300, bbox_inches="tight")

plt.show()


## 10. Introduction to MVMD

Environmental signals contain patterns occurring at different time scales. A variable may change rapidly from day to day while also following a slower seasonal pattern.

**Multivariate variational mode decomposition (MVMD)** separates several signals into simpler components, called **modes**.
```text
Complex Meteorological Signals (four variables together)
                          ↓
                         MVMD
                          ↓
              Mode 1 · Mode 2 · … · Mode K
                   Different time scales
```

Higher-frequency modes generally capture faster variations; lower-frequency modes generally capture slower variations. **Mode number alone does not tell us which is fastest.**

ETo is not decomposed. It remains tomorrow's answer for XGBoost to predict. Next, we load the supplied Python function and decompose training weather for an illustration.

## 11. Run the supplied MVMD implementation

The following cell is the **verbatim Python MVMD function from the source notebook**, translated there from the research MATLAB code. It is intentionally longer than the other cells: keeping the complete function makes this notebook self-contained and avoids replacing the method with another library. Participants can run this cell without reading every line.

This cell defines the algorithm; it does not yet run a decomposition. The next cells expose the settings and run it on meteorological predictors only.

In [ ]:

import numpy as np

def MVMD(signal, alpha=1000, tau=0, K=3,
         DC=0, init=1, tol=1e-7,
         max_iter=500, return_info=False):
    """
    Multivariate Variational Mode Decomposition (MVMD).

    Parameters
    ----------
    signal : array-like
        Input data: observations x channels.
    alpha : float
        Bandwidth constraint.
    tau : float
        Dual-ascent step size.
    K : int
        Number of modes per channel.
    DC : bool
        If True, fix the first mode at zero frequency.
    init : int
        0 = zero initialization
        1 = uniformly distributed frequencies
        2 = random frequencies
    tol : float
        Convergence tolerance.
    max_iter : int
        Maximum number of iterations.
    return_info : bool
        Return convergence information when True.

    Returns
    -------
    u : ndarray
        Shape: (K, observations, channels).
    u_hat : ndarray
        Fourier spectra of u.
    omega : ndarray
        Center-frequency history.
    residuals : ndarray
        Shape: (observations, channels).
        Original signal minus the sum of K modes.
    info : dict, optional
        Convergence and reconstruction information.
    """

    # --------------------------------------------------
    # 1. Input validation
    # --------------------------------------------------

    signal = np.asarray(signal, dtype=float)

    if signal.ndim != 2:
        raise ValueError(
            "Input must have shape (observations, channels)."
        )

    if not np.isfinite(signal).all():
        raise ValueError(
            "Input contains NaN or infinite values."
        )

    if int(K) != K or K < 1:
        raise ValueError("K must be a positive integer.")

    if alpha <= 0 or tol <= 0:
        raise ValueError(
            "alpha and tol must be positive."
        )

    if int(max_iter) != max_iter or max_iter < 1:
        raise ValueError(
            "max_iter must be a positive integer."
        )

    if tau < 0:
        raise ValueError("tau cannot be negative.")

    K = int(K)
    max_iter = int(max_iter)

    original_length, C = signal.shape

    if original_length < 4:
        raise ValueError(
            "At least four observations are required."
        )

    # Preserve the original data for residual calculation
    original_signal = signal.copy()

    # --------------------------------------------------
    # 2. Handle odd number of observations
    # --------------------------------------------------

    padded = original_length % 2 != 0

    if padded:
        signal = np.pad(
            signal,
            ((0, 1), (0, 0)),
            mode="symmetric"
        )

    L = signal.shape[0]
    half = L // 2

    # Channels x observations
    signal = signal.T

    # --------------------------------------------------
    # 3. Mirror extension
    # --------------------------------------------------

    f = np.concatenate(
        [
            signal[:, :half][:, ::-1],
            signal,
            signal[:, half:][:, ::-1]
        ],
        axis=1
    )

    T = f.shape[1]

    freqs = np.fft.fftshift(
        np.fft.fftfreq(T)
    )

    # Fourier transform
    f_hat = np.fft.fftshift(
        np.fft.fft(f, axis=1),
        axes=1
    ).T

    f_hat_plus = f_hat.copy()
    f_hat_plus[:T // 2, :] = 0

    # --------------------------------------------------
    # 4. Initialize modes and center frequencies
    # --------------------------------------------------

    modes = np.zeros(
        (T, C, K),
        dtype=complex
    )

    omega_history = np.zeros(
        (max_iter + 1, K),
        dtype=float
    )

    if init == 1:
        omega_history[0, :] = (
            0.5 / K
        ) * np.arange(K)

    elif init == 2:
        rng = np.random.default_rng(42)
        fs = 1.0 / L

        omega_history[0, :] = np.sort(
            np.exp(
                np.log(fs)
                + (
                    np.log(0.5) - np.log(fs)
                ) * rng.random(K)
            )
        )

    elif init != 0:
        raise ValueError(
            "init must be 0, 1, or 2."
        )

    if DC:
        omega_history[0, 0] = 0

    lambda_hat = np.zeros(
        (T, C),
        dtype=complex
    )

    converged = False
    relative_change = np.inf

    # --------------------------------------------------
    # 5. MVMD optimization
    # --------------------------------------------------

    for iteration in range(max_iter):

        old_modes = modes.copy()

        old_omega = (
            omega_history[iteration].copy()
        )

        new_omega = old_omega.copy()

        # Update each mode
        for k in range(K):

            # Sum of all other modes
            other_modes = (
                np.sum(modes, axis=2)
                - modes[:, :, k]
            )

            denominator = (
                1.0
                + alpha * (
                    freqs - old_omega[k]
                ) ** 2
            )

            modes[:, :, k] = (
                f_hat_plus
                - other_modes
                - lambda_hat / 2
            ) / denominator[:, None]

            # Update center frequency
            if DC and k == 0:
                new_omega[k] = 0.0

            else:
                positive = slice(
                    T // 2, T
                )

                power = np.abs(
                    modes[positive, :, k]
                ) ** 2

                total_power = np.sum(power)

                if total_power > 0:
                    new_omega[k] = np.sum(
                        freqs[positive, None]
                        * power
                    ) / total_power

        # Dual-ascent update
        lambda_hat += tau * (
            np.sum(modes, axis=2)
            - f_hat_plus
        )

        omega_history[
            iteration + 1
        ] = new_omega

        # Convergence check
        difference = np.linalg.norm(
            modes - old_modes
        )

        reference = max(
            np.linalg.norm(old_modes),
            1e-12
        )

        relative_change = (
            difference / reference
        )

        if relative_change < tol:
            converged = True
            break

    omega = omega_history[
        :iteration + 2
    ]

    # --------------------------------------------------
    # 6. Reconstruct full Fourier spectra
    # --------------------------------------------------

    full_spectra = np.zeros_like(
        modes
    )

    full_spectra[T // 2:, :, :] = (
        modes[T // 2:, :, :]
    )

    full_spectra[
        1:T // 2 + 1, :, :
    ] = np.conj(
        modes[
            T // 2:, :, :
        ][::-1, :, :]
    )

    full_spectra[0, :, :] = np.conj(
        full_spectra[-1, :, :]
    )

    # --------------------------------------------------
    # 7. Inverse Fourier transform
    # --------------------------------------------------

    reconstructed = np.real(
        np.fft.ifft(
            np.fft.ifftshift(
                full_spectra,
                axes=0
            ),
            axis=0
        )
    )

    # Remove mirrored observations
    reconstructed = reconstructed[
        T // 4:3 * T // 4, :, :
    ]

    # Remove padding if applied
    reconstructed = reconstructed[
        :original_length
    ]

    # Output shape:
    # modes x observations x channels
    u = np.transpose(
        reconstructed,
        (2, 0, 1)
    )

    # --------------------------------------------------
    # 8. Calculate Fourier spectra of modes
    # --------------------------------------------------

    u_hat = np.fft.fftshift(
        np.fft.fft(
            u,
            axis=1
        ),
        axes=1
    )

    # --------------------------------------------------
    # 9. Calculate residuals
    # --------------------------------------------------

    # Sum the K modes for every input channel
    reconstructed_signal = np.sum(
        u,
        axis=0
    )

    # Residual = original signal - reconstructed modes
    residuals = (
        original_signal - reconstructed_signal
    )

    # --------------------------------------------------
    # 10. Reconstruction accuracy
    # --------------------------------------------------

    reconstruction_error = (
        np.linalg.norm(residuals)
        / max(
            np.linalg.norm(original_signal),
            1e-12
        )
    )

    reconstruction_rmse = np.sqrt(
        np.mean(residuals ** 2)
    )

    # --------------------------------------------------
    # 11. Convergence information
    # --------------------------------------------------

    info = {
        "converged": converged,
        "iterations": iteration + 1,
        "relative_change": float(
            relative_change
        ),
        "padded": padded,
        "reconstruction_error": float(
            reconstruction_error
        ),
        "reconstruction_rmse": float(
            reconstruction_rmse
        )
    }

    # --------------------------------------------------
    # 12. Return results
    # --------------------------------------------------

    if return_info:
        return u, u_hat, omega, residuals, info

    return u, u_hat, omega, residuals


### Parameters and exogenous weather decomposition

We retain `alpha=1000`, `tau=0`, `DC=0`, `init=0`, and `tol=1e-7`. **K=3** modes is the setting that scored best on a validation block taken from the *end of the training period* (the test period was not used); it is provisional, not a proven optimum.

- Only the four weather drivers are decomposed. **ETo is never decomposed and never an input.**
- Modes are *not* given physical labels in advance. Read each mode's calculated center frequency (cycles/day) in the plot labels before describing its time scale.
- **The hybrid model's inputs are decomposed modes only.** No raw (un-decomposed) weather column is passed to it.



In [ ]:

import pandas as pd

# MVMD parameters
K = 3
alpha = 1000
tau = 0
DC = 0
init = 1
tol = 1e-7

# Prepare 24 lagged weather inputs
mvmd_source = model_df.sort_values("Date").copy()
weather_data = mvmd_source[feature_columns].to_numpy(dtype=float)

# Perform MVMD
u, u_hat, omega, residuals, info = MVMD(
    weather_data,
    alpha=alpha,
    tau=tau,
    K=K,
    DC=DC,
    init=init,
    tol=tol,
    return_info=True
)

# Build MVMD feature dataset
mvmd_features = {}

for j, feature in enumerate(feature_columns):

    for k in range(K):
        mvmd_features[
            f"{feature}_Mode_{k+1}"
        ] = u[k, :, j]

    mvmd_features[
        f"{feature}_Residual"
    ] = residuals[:, j]

mvmd_X = pd.DataFrame(
    mvmd_features,
    index=mvmd_source.index
)

mvmd_y = mvmd_source[target].copy()

# Chronological 75/25 split
split_index = int(len(mvmd_X) * 0.75)

X_train_mvmd = mvmd_X.iloc[:split_index].copy()
X_test_mvmd = mvmd_X.iloc[split_index:].copy()

y_train_mvmd = mvmd_y.iloc[:split_index].copy()
y_test_mvmd = mvmd_y.iloc[split_index:].copy()

# Verification
expected_features = len(feature_columns) * (K + 1)

assert mvmd_X.shape[1] == expected_features
assert len(mvmd_X) == len(mvmd_y)

print("MVMD completed")
print("Converged:", info["converged"])
print("Iterations:", info["iterations"])
print("Reconstruction RMSE:", info["reconstruction_rmse"])

print("\nOriginal weather features:", len(feature_columns))
print("Mode features:", len(feature_columns) * K)
print("Residual features:", len(feature_columns))
print("Total MVMD-XGBoost features:", mvmd_X.shape[1])

print("\nTraining:", X_train_mvmd.shape)
print("Testing:", X_test_mvmd.shape)

display(mvmd_X.head())



### Understanding the Number of Input Features: 24, 72, and 96

The number of input features depends on the number of meteorological variables, selected time steps, and decomposition components.

**1. Standalone XGBoost — 24 Features**

We use:
- 4 meteorological variables
- 6 time steps (t through t−5)

**Total: 4 × 6 = 24 features**

**2. MVMD-XGBoost — 72 Features (Modes Only)**

MVMD decomposes each meteorological variable into 3 modes.

We use:
- 4 meteorological variables
- 3 modes per variable
- 6 time steps

**Total: 4 × 3 × 6 = 72 features**

**3. MVMD-XGBoost — 96 Features (Modes + Residual)**

When we also include the residual component, each variable has 4 components: 3 modes and 1 residual.

We use:
- 4 meteorological variables
- 4 components per variable
- 6 time steps

**Total: 4 × 4 × 6 = 96 features**

### Why does this matter?

The standalone XGBoost model uses the original weather observations, while MVMD-XGBoost uses the decomposed components of those observations.

Including the residual increases the number of features from 72 to 96.

**In our workshop, we use 96 features for MVMD-XGBoost because we include both the three modes and the residual component.**


## 12. Visualize MVMD without plot overload

We will inspect **solar radiation** in a single stacked figure: the original signal, then Modes 1 through K. Expect smoother and faster-changing components in the same units as the original variable. The row labels show the actual center frequencies; compare them before interpreting a mode's time scale.

This illustration helps explain what decomposition provides to the hybrid model. Each row has its own vertical scale to keep the components visible, so use the tick labels when comparing amplitudes.

> **Try it:** change `inspect_variable` to another name in `predictors` and rerun only this plot. This changes the illustration, not the fitted models.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# =====================================================
# 1. Select meteorological variable and time lag
# =====================================================

inspect_variable = "Solar_Radiation_Wm2"
inspect_lag = 0  # 0=t, 1=t-1, ..., 5=t-5

# Identify feature among the 24 lagged inputs
feature_name = f"{inspect_variable}_t-{inspect_lag}"

assert inspect_variable in predictors
assert 0 <= inspect_lag <= 5
assert feature_name in feature_columns

channel = feature_columns.index(feature_name)

# =====================================================
# 2. Prepare dates, original signal, modes, residual
# =====================================================

# Forecast-origin dates (t)
# For lag > 0, the selected input was observed earlier.
mode_dates = pd.to_datetime(
    mvmd_source["Origin_Date"]
).to_numpy()

original_signal = weather_data[:, channel]

# Number of MVMD modes
n_modes = u.shape[0]

# Select the residual for this channel
residual_signal = residuals[:, channel]

# Verify dimensions
assert len(mode_dates) == len(original_signal)
assert u.shape[1] == len(original_signal)
assert u.shape[2] == len(feature_columns)
assert residuals.shape == weather_data.shape

# Units
units = {
    "Solar_Radiation_Wm2": "W/m²",
    "Air_Temp_C": "°C",
    "Relative_Humidity_pct": "%",
    "Wind_Speed_ms": "m/s"
}

variable_label = variable_names.get(
    inspect_variable,
    inspect_variable
)

unit_label = units.get(inspect_variable, "")

lag_label = (
    "t" if inspect_lag == 0
    else f"t−{inspect_lag}"
)

# =====================================================
# 3. Check reconstruction accuracy
# =====================================================

# Reconstruct original signal using modes + residual
reconstructed_signal = (
    np.sum(u[:, :, channel], axis=0)
    + residual_signal
)

reconstruction_rmse = np.sqrt(
    np.mean(
        (original_signal - reconstructed_signal) ** 2
    )
)

print("Selected feature:", feature_name)
print("Number of MVMD modes:", n_modes)
print("Residual components: 1")
print("Total components:", n_modes + 1)
print(
    "Reconstruction RMSE:",
    f"{reconstruction_rmse:.10f}"
)

# =====================================================
# 4. Create figure
# =====================================================

# Original + K modes + residual
n_panels = n_modes + 2

fig, axes = plt.subplots(
    n_panels,
    1,
    figsize=(13, 2.0 * n_panels),
    sharex=True
)

mode_colors = [
    "#0072B2",
    "#D55E00",
    "#009E73",
    "#CC79A7",
    "#E69F00",
    "#56B4E9"
]

# =====================================================
# 5. Plot original meteorological signal
# =====================================================

axes[0].plot(
    mode_dates,
    original_signal,
    color="#333333",
    linewidth=1.0
)

axes[0].set_ylabel(
    "Original",
    fontsize=10,
    fontweight="bold"
)

axes[0].grid(alpha=0.2)

axes[0].set_title(
    f"MVMD Decomposition — {variable_label} "
    f"({lag_label}, {unit_label})",
    fontsize=14,
    fontweight="bold",
    pad=12
)

# =====================================================
# 6. Plot K MVMD modes
# =====================================================

for k in range(n_modes):

    ax = axes[k + 1]

    ax.plot(
        mode_dates,
        u[k, :, channel],
        color=mode_colors[k % len(mode_colors)],
        linewidth=1.0
    )

    ax.set_ylabel(
        f"Mode {k + 1}",
        fontsize=10,
        fontweight="bold"
    )

    ax.grid(alpha=0.2)

    # Center frequency of each MVMD mode
    ax.text(
        0.99,
        0.88,
        f"Frequency = {omega[-1, k]:.4f} cycles/day",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=9,
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.85
        )
    )

# =====================================================
# 7. Plot residual component
# =====================================================

ax_residual = axes[-1]

ax_residual.plot(
    mode_dates,
    residual_signal,
    color="#C62828",
    linewidth=1.0
)

ax_residual.axhline(
    0,
    color="black",
    linewidth=0.7,
    linestyle="--",
    alpha=0.6
)

ax_residual.set_ylabel(
    "Residual",
    fontsize=10,
    fontweight="bold"
)

ax_residual.grid(alpha=0.2)

ax_residual.text(
    0.99,
    0.88,
    "Residual = Original − Sum of Modes",
    transform=ax_residual.transAxes,
    ha="right",
    va="top",
    fontsize=9,
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.85
    )
)

# =====================================================
# 8. Format date axis
# =====================================================

date_locator = mdates.AutoDateLocator()

axes[-1].xaxis.set_major_locator(
    date_locator
)

axes[-1].xaxis.set_major_formatter(
    mdates.ConciseDateFormatter(date_locator)
)

axes[-1].set_xlabel(
    "Forecast-Origin Date",
    fontsize=11
)

for ax in axes:
    ax.tick_params(
        axis="both",
        labelsize=9
    )

# =====================================================
# 9. Adjust spacing
# =====================================================

plt.tight_layout(h_pad=1.1)

# =====================================================
# 10. Save figure
# =====================================================

output_name = (
    f"MVMD_{inspect_variable}_lag{inspect_lag}"
    f"_with_residual.png"
)

plt.savefig(
    output_name,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nFigure saved as:", output_name)


### Sequential mode animation

Each mode of the training-period decomposition is revealed in turn, with the running sum of modes compared against the observed solar radiation. The statistics box reports how closely the partial sum reconstructs the signal; a decomposition reconstructs its own input by construction, so this shows what the modes are, **not** how well they forecast.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

# =====================================================
# 1. Select meteorological variable and time lag
# =====================================================

inspect_variable = "Solar_Radiation_Wm2"
inspect_lag = 0  # 0=t, 1=t-1, ..., 5=t-5

feature_name = f"{inspect_variable}_t-{inspect_lag}"

if feature_name not in feature_columns:
    raise ValueError(f"Feature not found: {feature_name}")

anim_ch = feature_columns.index(feature_name)

# =====================================================
# 2. Prepare original signal, modes, and residual
# =====================================================

raw_anim = np.asarray(
    weather_data[:, anim_ch], dtype=float
)

n_modes = u.shape[0]

modes_anim = [
    np.asarray(u[k, :, anim_ch], dtype=float)
    for k in range(n_modes)
]

residual_anim = np.asarray(
    residuals[:, anim_ch], dtype=float
)

components_anim = modes_anim + [residual_anim]

component_names = (
    [f"Mode {k + 1}" for k in range(n_modes)]
    + ["Residual"]
)

n_components = len(components_anim)

# Forecast-origin dates
dates_series = pd.to_datetime(
    mvmd_source["Origin_Date"]
).reset_index(drop=True)

assert len(raw_anim) == len(dates_series)
assert all(
    len(component) == len(raw_anim)
    for component in components_anim
)

# =====================================================
# 3. Downsample for animation performance
# =====================================================

ds_step = 4

t_idx = np.arange(
    0, len(raw_anim), ds_step
)

anim_dates = dates_series.iloc[t_idx].to_numpy()
raw_ds = raw_anim[t_idx]

components_ds = np.asarray([
    component[t_idx]
    for component in components_anim
])

N_pts = len(raw_ds)

# =====================================================
# 4. Verify reconstruction
# =====================================================

full_reconstruction = np.sum(
    components_ds, axis=0
)

final_error = raw_ds - full_reconstruction

full_rmse = np.sqrt(
    np.mean(final_error ** 2)
)

# Reconstruction before adding the residual
modes_only_reconstruction = np.sum(
    components_ds[:n_modes], axis=0
)

modes_only_rmse = np.sqrt(
    np.mean(
        (raw_ds - modes_only_reconstruction) ** 2
    )
)

print("Selected feature:", feature_name)
print("Original observations:", len(raw_anim))
print("MVMD modes:", n_modes)
print("Residual components:", 1)
print("Total components:", n_components)
print(f"RMSE using modes only: {modes_only_rmse:.4f}")
print(f"RMSE including residual: {full_rmse:.10f}")

# =====================================================
# 5. Figure settings
# =====================================================

plt.style.use("default")
plt.rcParams["animation.embed_limit"] = 100.0

fig = plt.figure(
    figsize=(13, 8),
    dpi=80
)

gs = gridspec.GridSpec(
    n_components,
    2,
    width_ratios=[1.35, 1.0],
    wspace=0.35,
    hspace=0.55
)

# Left: reconstruction and reconstruction error
ax_recon = fig.add_subplot(
    gs[:n_components - 1, 0]
)

ax_err = fig.add_subplot(
    gs[n_components - 1, 0],
    sharex=ax_recon
)

# Right: individual modes and residual
component_axes = [
    fig.add_subplot(gs[i, 1], sharex=ax_recon)
    for i in range(n_components)
]

palette = [
    "#0072B2",
    "#D55E00",
    "#009E73",
    "#CC79A7",
    "#E69F00",
    "#56B4E9"
]

units = {
    "Solar_Radiation_Wm2": "W/m²",
    "Air_Temp_C": "°C",
    "Relative_Humidity_pct": "%",
    "Wind_Speed_ms": "m/s"
}

unit_label = units.get(inspect_variable, "")

variable_label = variable_names.get(
    inspect_variable, inspect_variable
)

lag_label = (
    "t" if inspect_lag == 0
    else f"t−{inspect_lag}"
)

# =====================================================
# 6. Original and reconstructed signal
# =====================================================

ax_recon.plot(
    anim_dates,
    raw_ds,
    color="#666666",
    alpha=0.7,
    linewidth=1.2,
    linestyle="--",
    label="Original signal"
)

recon_line, = ax_recon.plot(
    [],
    [],
    color="#2E7D32",
    linewidth=2.0,
    label="Cumulative reconstruction"
)

recon_pt, = ax_recon.plot(
    [],
    [],
    "o",
    color="#2E7D32",
    markersize=4
)

# Determine limits using all cumulative stages
cumulative_signals = np.cumsum(
    components_ds, axis=0
)

all_values = np.concatenate([
    raw_ds,
    cumulative_signals.ravel()
])

signal_pad = max(
    0.1 * np.ptp(all_values), 1
)

ax_recon.set_ylim(
    np.min(all_values) - signal_pad,
    np.max(all_values) + signal_pad
)

ax_recon.set_xlim(
    anim_dates[0],
    anim_dates[-1]
)

ax_recon.set_ylabel(
    f"{variable_label} ({unit_label})",
    fontsize=9,
    fontweight="bold"
)

ax_recon.set_title(
    f"Sequential MVMD Reconstruction\n"
    f"{variable_label} ({lag_label})",
    fontsize=12,
    fontweight="bold",
    pad=12
)

ax_recon.legend(
    loc="upper right",
    fontsize=8
)

ax_recon.grid(
    linestyle=":",
    alpha=0.3
)

stats_box = ax_recon.text(
    0.02,
    0.96,
    "",
    transform=ax_recon.transAxes,
    fontsize=9,
    va="top",
    bbox=dict(
        boxstyle="round,pad=0.4",
        facecolor="white",
        edgecolor="#CCCCCC",
        alpha=0.95
    )
)

# =====================================================
# 7. Reconstruction error panel
# =====================================================

err_line, = ax_err.plot(
    [],
    [],
    color="#C62828",
    linewidth=1.3,
    label="Reconstruction error"
)

partial_errors = (
    raw_ds[None, :] - cumulative_signals
)

max_error = max(
    np.max(np.abs(partial_errors)) * 1.15,
    1
)

ax_err.set_ylim(
    -max_error,
    max_error
)

ax_err.axhline(
    0,
    color="black",
    linewidth=0.7,
    linestyle="--"
)

ax_err.set_ylabel(
    f"Error ({unit_label})",
    fontsize=9,
    fontweight="bold"
)

ax_err.set_xlabel(
    "Forecast-Origin Date",
    fontsize=9,
    fontweight="bold"
)

ax_err.grid(
    linestyle=":",
    alpha=0.3
)

ax_err.legend(
    loc="upper right",
    fontsize=8
)

# =====================================================
# 8. Individual mode and residual panels
# =====================================================

component_lines = []
component_dots = []

for i, ax in enumerate(component_axes):

    color = (
        "#C62828"
        if i == n_modes
        else palette[i % len(palette)]
    )

    # Faint full component for reference
    ax.plot(
        anim_dates,
        components_ds[i],
        color=color,
        alpha=0.18,
        linewidth=1
    )

    # Animated component
    line, = ax.plot(
        [],
        [],
        color=color,
        linewidth=1.7
    )

    dot, = ax.plot(
        [],
        [],
        "o",
        color=color,
        markersize=4
    )

    component_lines.append(line)
    component_dots.append(dot)

    component_range = np.ptp(
        components_ds[i]
    )

    pad = max(
        0.15 * component_range,
        0.1
    )

    ax.set_ylim(
        np.min(components_ds[i]) - pad,
        np.max(components_ds[i]) + pad
    )

    ax.set_ylabel(
        component_names[i],
        fontsize=9,
        fontweight="bold",
        color=color
    )

    if i < n_modes:
        label_text = (
            f"{omega[-1, i]:.4f} cycles/day"
        )
    else:
        label_text = "Residual component"

    ax.text(
        0.98,
        0.90,
        label_text,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=8,
        color=color,
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.8
        )
    )

    ax.grid(
        linestyle=":",
        alpha=0.3
    )

    if i == n_components - 1:
        ax.set_xlabel(
            "Forecast-Origin Date",
            fontsize=9
        )
    else:
        ax.tick_params(
            labelbottom=False
        )

# =====================================================
# 9. Format date axes
# =====================================================

date_locator = mdates.AutoDateLocator()

ax_err.xaxis.set_major_locator(
    date_locator
)

ax_err.xaxis.set_major_formatter(
    mdates.ConciseDateFormatter(date_locator)
)

for ax in [ax_recon, ax_err] + component_axes:
    ax.tick_params(
        axis="both",
        labelsize=8
    )

# =====================================================
# 10. Animation settings
# =====================================================

frames_per_component = 20
pause_frames = 6

cycle = frames_per_component + pause_frames

total_anim_frames = n_components * cycle

# =====================================================
# 11. Animation update function
# =====================================================

def update_anim(frame):

    active_component = min(
        frame // cycle,
        n_components - 1
    )

    sub_frame = frame % cycle

    progress = min(
        (sub_frame + 1) / frames_per_component,
        1.0
    )

    s = max(
        1,
        min(int(progress * N_pts), N_pts)
    )

    # Update individual components
    for i in range(n_components):

        if i < active_component:

            component_lines[i].set_data(
                anim_dates,
                components_ds[i]
            )

            component_dots[i].set_data([], [])

        elif i == active_component:

            component_lines[i].set_data(
                anim_dates[:s],
                components_ds[i][:s]
            )

            component_dots[i].set_data(
                [anim_dates[s - 1]],
                [components_ds[i, s - 1]]
            )

        else:

            component_lines[i].set_data([], [])
            component_dots[i].set_data([], [])

    # Cumulative reconstruction:
    # previously completed components + active component
    current_reconstruction = np.zeros(N_pts)

    if active_component > 0:
        current_reconstruction += np.sum(
            components_ds[:active_component],
            axis=0
        )

    current_reconstruction[:s] += (
        components_ds[active_component, :s]
    )

    recon_line.set_data(
        anim_dates,
        current_reconstruction
    )

    recon_pt.set_data(
        [anim_dates[s - 1]],
        [current_reconstruction[s - 1]]
    )

    # Reconstruction error
    current_error = (
        raw_ds - current_reconstruction
    )

    err_line.set_data(
        anim_dates,
        current_error
    )

    # Reconstruction metrics
    rmse = np.sqrt(
        np.mean(current_error ** 2)
    )

    ss_res = np.sum(
        current_error ** 2
    )

    ss_tot = np.sum(
        (raw_ds - np.mean(raw_ds)) ** 2
    )

    r2 = (
        1 - ss_res / ss_tot
        if ss_tot > 0
        else np.nan
    )

    stats_box.set_text(
        f"Active: {component_names[active_component]}\n"
        f"Components: {active_component + 1}/{n_components}\n"
        f"Reconstruction R²: {r2:.4f}\n"
        f"RMSE: {rmse:.2f} {unit_label}"
    )

    return (
        component_lines
        + component_dots
        + [
            recon_line,
            recon_pt,
            err_line,
            stats_box
        ]
    )

# =====================================================
# 12. Create and display animation
# =====================================================

anim = FuncAnimation(
    fig,
    update_anim,
    frames=total_anim_frames,
    interval=80,
    blit=False,
    repeat=True
)

plt.close(fig)

display(HTML(anim.to_jshtml()))


## 13. Hybrid model — decomposed inputs only




### Train the hybrid on decomposed inputs

The hybrid uses the **same XGBoost hyperparameters as the baseline**, so any difference in score comes from the inputs, not from tuning. It maps mode features to the actual next-day ETo and is evaluated on exactly the baseline's test dates.

In [ ]:

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

# ============================================
# 1. Verify MVMD features
# ============================================

print("Training features:", X_train_mvmd.shape)
print("Testing features:", X_test_mvmd.shape)

assert X_train_mvmd.shape[1] == 96
assert X_test_mvmd.shape[1] == 96

assert not X_train_mvmd.isna().any().any()
assert not X_test_mvmd.isna().any().any()

# ============================================
# 2. Initialize XGBoost
# ============================================

mvmd_xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# ============================================
# 3. Train model
# ============================================

mvmd_xgb_model.fit(
    X_train_mvmd,
    y_train_mvmd
)

# ============================================
# 4. Generate predictions
# ============================================

y_train_pred_mvmd = mvmd_xgb_model.predict(
    X_train_mvmd
)

y_test_pred_mvmd = mvmd_xgb_model.predict(
    X_test_mvmd
)

print("\nMVMD-XGBoost training completed!")
print("Training predictions:", len(y_train_pred_mvmd))
print("Testing predictions:", len(y_test_pred_mvmd))


In [ ]:

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ============================================
# 1. Performance function
# ============================================

def calculate_metrics(y_true, y_pred):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mae = mean_absolute_error(y_true, y_pred)

    r2 = r2_score(y_true, y_pred)

    r = np.corrcoef(
        y_true, y_pred
    )[0, 1]

    nse = 1 - (
        np.sum((y_true - y_pred) ** 2)
        /
        np.sum((y_true - np.mean(y_true)) ** 2)
    )

    return {
        "R": r,
        "RMSE (mm/day)": rmse,
        "MAE (mm/day)": mae,
        "R²": r2,
        "NSE": nse
    }

# ============================================
# 2. Training and testing performance
# ============================================

train_metrics_mvmd = calculate_metrics(
    y_train_mvmd,
    y_train_pred_mvmd
)

test_metrics_mvmd = calculate_metrics(
    y_test_mvmd,
    y_test_pred_mvmd
)

# ============================================
# 3. Performance table
# ============================================

performance_mvmd = pd.DataFrame(
    [
        train_metrics_mvmd,
        test_metrics_mvmd
    ],
    index=["Training", "Testing"]
)

print("MVMD-XGBoost Performance")
display(performance_mvmd.round(4))


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Predict ETo for MVMD-XGBoost training and testing datasets
y_train_pred_mvmd = mvmd_xgb_model.predict(X_train_mvmd)
y_test_pred_mvmd = mvmd_xgb_model.predict(X_test_mvmd)

# Convert observations to arrays
y_train_true_mvmd = np.asarray(y_train_mvmd).flatten()
y_test_true_mvmd = np.asarray(y_test_mvmd).flatten()

# Calculate Pearson correlation coefficients
R_train_mvmd = np.corrcoef(
    y_train_true_mvmd, y_train_pred_mvmd
)[0, 1]

R_test_mvmd = np.corrcoef(
    y_test_true_mvmd, y_test_pred_mvmd
)[0, 1]

# Create scatter plot
plt.figure(figsize=(8, 7))

plt.scatter(y_train_true_mvmd, y_train_pred_mvmd,
            color="blue", alpha=0.5, s=30,
            label="Training")

plt.scatter(y_test_true_mvmd, y_test_pred_mvmd,
            color="red", alpha=0.6, s=30,
            marker="x", label="Testing")

# Plot 1:1 reference line
min_val = min(
    y_train_true_mvmd.min(),
    y_test_true_mvmd.min(),
    y_train_pred_mvmd.min(),
    y_test_pred_mvmd.min()
)

max_val = max(
    y_train_true_mvmd.max(),
    y_test_true_mvmd.max(),
    y_train_pred_mvmd.max(),
    y_test_pred_mvmd.max()
)

plt.plot([min_val, max_val], [min_val, max_val],
         "k--", label="1:1 Line")

# Display correlation coefficients
plt.text(0.05, 0.92, f"R (Train) = {R_train_mvmd:.3f}",
         transform=plt.gca().transAxes,
         fontsize=11, color="blue")

plt.text(0.05, 0.86, f"R (Test) = {R_test_mvmd:.3f}",
         transform=plt.gca().transAxes,
         fontsize=11, color="red")

# Figure settings
plt.xlabel("Observed ETo (mm/day)")
plt.ylabel("Predicted ETo (mm/day)")
plt.title("Observed vs. Predicted ETo — MVMD-XGBoost (1-Day Ahead)")

plt.xlim(min_val, max_val)
plt.ylim(min_val, max_val)
plt.gca().set_aspect("equal", adjustable="box")

plt.legend(loc="lower right")
plt.grid(alpha=0.2)
plt.tight_layout()

# Save figure
plt.savefig("MVMD_XGBoost_Observed_vs_Predicted.png",
            dpi=300, bbox_inches="tight")

plt.show()


## 14. Compare standalone and hybrid models

Both models are scored on the same held-out test dates. A third row shows the standalone model trained on the *same training rows* as the hybrid, so the comparison is not affected by the hybrid having to drop its earliest rows.

- **Standalone XGBoost:** raw lagged weather → tomorrow's ETo.
- **Hybrid MVMD-XGBoost:** MVMD modes of the weather only → tomorrow's ETo.

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# =====================================================
# 1. Get standalone and hybrid predictions
# =====================================================

# Standalone XGBoost: 24 original weather features
standalone_pred = xgb_model.predict(X_test)

# Hybrid MVMD-XGBoost: 96 decomposed features
hybrid_pred = mvmd_xgb_model.predict(X_test_mvmd)

# Observed values
standalone_true = np.asarray(y_test).flatten()
hybrid_true = np.asarray(y_test_mvmd).flatten()

# =====================================================
# 2. Verify identical testing observations
# =====================================================

# Both models must use the same target dates and ETo
# observations for a fair numerical comparison.

assert len(standalone_true) == len(hybrid_true), (
    "The models have different numbers of test observations."
)

assert np.allclose(standalone_true, hybrid_true), (
    "The testing ETo observations are not aligned."
)

assert len(standalone_pred) == len(hybrid_pred)

# =====================================================
# 3. Calculate performance metrics
# =====================================================

def score_forecast(y_true, y_pred):

    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    mae = mean_absolute_error(y_true, y_pred)

    r2 = r2_score(y_true, y_pred)

    r = np.corrcoef(
        y_true, y_pred
    )[0, 1]

    nse = 1 - (
        np.sum((y_true - y_pred) ** 2) /
        np.sum((y_true - np.mean(y_true)) ** 2)
    )

    return {
        "R": r,
        "RMSE": rmse,
        "MAE": mae,
        "R²": r2,
        "NSE": nse
    }

# =====================================================
# 4. Compare testing performance
# =====================================================

model_names = [
    "Standalone XGBoost",
    "Hybrid MVMD-XGBoost"
]

baseline_metrics = score_forecast(
    standalone_true,
    standalone_pred
)

hybrid_metrics = score_forecast(
    hybrid_true,
    hybrid_pred
)

comparison = pd.DataFrame(
    [baseline_metrics, hybrid_metrics],
    index=model_names
)

comparison.index.name = "Model"

print("Testing Performance Comparison")
display(comparison.round(4))

# =====================================================
# 5. Calculate improvements
# =====================================================

def pct_reduction(base, new):
    if base == 0:
        return np.nan
    return (base - new) / base * 100

rmse_reduction = pct_reduction(
    baseline_metrics["RMSE"],
    hybrid_metrics["RMSE"]
)

mae_reduction = pct_reduction(
    baseline_metrics["MAE"],
    hybrid_metrics["MAE"]
)

r2_gain = (
    hybrid_metrics["R²"] -
    baseline_metrics["R²"]
)

print("\nHybrid MVMD-XGBoost vs. Standalone XGBoost")

print(
    f"RMSE: {hybrid_metrics['RMSE']:.4f} vs. "
    f"{baseline_metrics['RMSE']:.4f} mm/day "
    f"({rmse_reduction:+.2f}% reduction)"
)

print(
    f"MAE: {hybrid_metrics['MAE']:.4f} vs. "
    f"{baseline_metrics['MAE']:.4f} mm/day "
    f"({mae_reduction:+.2f}% reduction)"
)

print(
    f"R²: {hybrid_metrics['R²']:.4f} vs. "
    f"{baseline_metrics['R²']:.4f} "
    f"({r2_gain:+.4f} difference)"
)

# =====================================================
# 6. Visualization: RMSE and MAE comparison
# =====================================================

colors = [
    "#0072B2",  # Standalone XGBoost
    "#D55E00"   # Hybrid MVMD-XGBoost
]

fig, ax = plt.subplots(
    figsize=(10, 4.5)
)

metrics_to_plot = ["RMSE", "MAE"]

for row, metric in enumerate(metrics_to_plot):

    values = [
        comparison.loc[name, metric]
        for name in model_names
    ]

    # Connecting line
    ax.plot(
        values,
        [row, row],
        color="#BDC5CC",
        linewidth=4,
        solid_capstyle="round",
        zorder=1
    )

    # Model markers and values
    for i, name in enumerate(model_names):

        val = float(
            comparison.loc[name, metric]
        )

        ax.scatter(
            val,
            row,
            s=120,
            color=colors[i],
            edgecolors="white",
            linewidths=1.2,
            label=name if row == 0 else None,
            zorder=3
        )

        ax.annotate(
            f"{val:.3f} mm/day",
            (val, row),
            xytext=(0, 15 if i == 0 else -25),
            textcoords="offset points",
            ha="center",
            fontsize=11,
            fontweight="bold",
            color=colors[i]
        )

ax.set(
    yticks=[0, 1],
    yticklabels=["RMSE", "MAE"],
    ylim=(1.6, -0.6),
    xlim=(
        0,
        float(
            comparison[["RMSE", "MAE"]]
            .to_numpy().max()
        ) * 1.25
    ),
    xlabel="Error (mm/day) · Lower is better",
    title="ETo Prediction Error — Standalone vs. Hybrid MVMD-XGBoost"
)

ax.grid(False, axis="y")

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=2,
    frameon=False,
    fontsize=11
)

fig.tight_layout()

plt.savefig(
    "XGBoost_vs_MVMD_XGBoost_Comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


### Observed versus predicted ETo — identical axes

Both models on identical axes with a 1:1 reference line. Compare how tightly each cloud follows the diagonal and whether either misses systematically at high or low ETo.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# =====================================================
# 1. Prepare testing observations and predictions
# =====================================================

standalone_true = np.asarray(y_test).flatten()
hybrid_true = np.asarray(y_test_mvmd).flatten()

standalone_pred = xgb_model.predict(X_test)
hybrid_pred = mvmd_xgb_model.predict(X_test_mvmd)

# Verify the same testing observations
assert len(standalone_true) == len(hybrid_true)
assert np.allclose(standalone_true, hybrid_true), (
    "Testing observations are not aligned."
)

model_names = [
    "Standalone XGBoost",
    "Hybrid MVMD-XGBoost"
]

test_predictions = {
    "Standalone XGBoost": standalone_pred,
    "Hybrid MVMD-XGBoost": hybrid_pred
}

# Bright red and bright green
colors = [
    "#FF3030",  # Bright red
    "#00C853"   # Bright green
]

# =====================================================
# 2. Shared scatter plot limits
# =====================================================

all_scatter_values = np.concatenate([
    standalone_true,
    standalone_pred,
    hybrid_pred
])

scatter_low = min(
    0.0,
    float(all_scatter_values.min())
)

scatter_high = float(
    all_scatter_values.max()
)

scatter_pad = max(
    0.2,
    (scatter_high - scatter_low) * 0.05
)

shared_limits = (
    scatter_low - scatter_pad,
    scatter_high + scatter_pad
)

# =====================================================
# 3. Create side-by-side scatter plots
# =====================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 6),
    sharex=True,
    sharey=True
)

for i, (ax, name) in enumerate(
    zip(axes, model_names)
):

    pred_vals = test_predictions[name]

    # Calculate performance
    m = score_forecast(
        standalone_true,
        pred_vals
    )

    # Scatter points
    ax.scatter(
        standalone_true,
        pred_vals,
        s=30,
        color=colors[i],
        alpha=0.65,
        edgecolors="none"
    )

    # 1:1 reference line
    ax.plot(
        shared_limits,
        shared_limits,
        "--",
        color="#444444",
        linewidth=1.3,
        label="1:1 Line"
    )

    # Figure settings
    ax.set(
        xlim=shared_limits,
        ylim=shared_limits,
        xlabel="Observed ETo (mm/day)",
        title=name
    )

    ax.set_aspect(
        "equal",
        adjustable="box"
    )

    ax.set_title(
        name,
        fontsize=13,
        fontweight="bold",
        color=colors[i],
        pad=12
    )

    # Performance annotation
    annotation = (
        f"R      : {m['R']:.3f}\n"
        f"RMSE : {m['RMSE']:.3f} mm/day\n"
        f"MAE   : {m['MAE']:.3f} mm/day\n"
        f"R²      : {m['R²']:.3f}\n"
        f"NSE   : {m['NSE']:.3f}"
    )

    ax.text(
        0.05,
        0.95,
        annotation,
        transform=ax.transAxes,
        va="top",
        fontsize=11,
        bbox=dict(
            facecolor="white",
            edgecolor="#E0E4E8",
            alpha=0.94,
            boxstyle="round,pad=0.5"
        )
    )

    ax.grid(
        alpha=0.2,
        linestyle=":"
    )

    ax.legend(
        loc="lower right",
        fontsize=9
    )

# =====================================================
# 4. Final figure settings
# =====================================================

axes[0].set_ylabel(
    "Predicted ETo (mm/day)",
    fontsize=11
)

fig.suptitle(
    "Observed vs. Predicted ETo — Davis Held-Out Test Period",
    fontsize=15,
    fontweight="bold",
    y=1.02
)

fig.tight_layout()

# Save high-resolution figure
plt.savefig(
    "Standalone_vs_MVMD_XGBoost_Scatter.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


### Violin plot — prediction bias and error distribution

Distribution of signed errors (predicted − observed). A narrower violin centered on zero means smaller and less biased errors.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# ============================================
# 1. Prepare prediction errors
# ============================================

observed = np.asarray(y_test).flatten()

standalone_pred = np.asarray(
    xgb_model.predict(X_test)
).flatten()

hybrid_pred = np.asarray(
    mvmd_xgb_model.predict(X_test_mvmd)
).flatten()

# Verify testing observations are aligned
assert np.allclose(
    observed,
    np.asarray(y_test_mvmd).flatten()
), "Testing observations are not aligned."

errors = [
    standalone_pred - observed,
    hybrid_pred - observed
]

model_names = [
    "Standalone XGBoost",
    "Hybrid MVMD-XGBoost"
]

# Neon red and green
colors = [
    "#FF1744",  # Neon red
    "#00E676"   # Neon green
]

# Darker colors for individual dots
dot_colors = [
    "#B00020",  # Dark red
    "#008F46"   # Dark green
]

# ============================================
# 2. Create violin plot
# ============================================

fig, ax = plt.subplots(figsize=(10, 5.5))

violins = ax.violinplot(
    errors,
    positions=[0, 1],
    widths=0.75,
    showmeans=False,
    showmedians=False,
    showextrema=False
)

rng = np.random.default_rng(42)

for i, (body, error) in enumerate(
    zip(violins["bodies"], errors)
):

    # Neon violin
    body.set_facecolor(colors[i])
    body.set_edgecolor(colors[i])
    body.set_linewidth(1.5)
    body.set_alpha(0.30)

    # Darker individual dots
    jitter = rng.uniform(
        -0.12, 0.12, len(error)
    )

    ax.scatter(
        i + jitter,
        error,
        s=17,
        alpha=0.65,
        color=dot_colors[i],
        edgecolors="none",
        zorder=3
    )

    # Quartiles and median
    q25, med, q75 = np.percentile(
        error, [25, 50, 75]
    )

    ax.plot(
        [i, i],
        [q25, q75],
        color="#333333",
        linewidth=4,
        solid_capstyle="round",
        zorder=4
    )

    ax.scatter(
        i,
        med,
        color="white",
        edgecolor="#222222",
        linewidth=1.5,
        s=60,
        zorder=5
    )

# ============================================
# 3. Figure settings
# ============================================

ax.axhline(
    0,
    color="#555555",
    linestyle="--",
    linewidth=1.2
)

ax.set_xticks([0, 1])
ax.set_xticklabels(
    model_names,
    fontsize=11
)

ax.set_ylabel(
    "Prediction Error (Predicted − Observed, mm/day)",
    fontsize=11
)

ax.set_title(
    "Prediction Error Distribution — Davis Test Period",
    fontsize=14,
    fontweight="bold",
    pad=12
)

ax.grid(
    axis="y",
    alpha=0.2,
    linestyle=":"
)

ax.set_axisbelow(True)

plt.tight_layout()

# Save figure
plt.savefig(
    "XGBoost_vs_MVMD_Error_Distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()



## Workshop Conclusions

### Key Takeaways

1. **Data Preparation:** Clean and organize environmental time-series data before modeling.

2. **Lag Identification:** Use XCF and PXCF to explore relationships at different time lags.

3. **XGBoost:** Forecast one-day-ahead ETo using lagged weather observations.

4. **MVMD:** Decompose multiple weather signals into frequency-aligned components.

5. **Model Comparison:** Evaluate forecasting performance using RMSE, MAE, R², and R.



---

### Thank You!

**Questions & Discussion**
